# [실습 12-2] 프롬프트 실험 — Before/After 비교 (2순위 경로: Colab)

| 항목 | 내용 |
|---|---|
| 예상 소요 시간 | 30~40분 (**GPU 권장** — 무료 T4로 충분) |
| 본문 연계 | 12.3 프롬프트 |
| 선수 실습 | [실습 12-1] (모델 로드 감각) |
| 비용 | **완전 무료** — 유료 API 키 불요(확정 사항 ④) |

> **경로 안내**: 이 실습의 **1순위 경로는 무료 웹 챗봇**이다 — 책 지면의
> 프롬프트 쌍을 챗봇에 직접 입력해 비교하면 코드가 필요 없다(최신 무료
> 서비스 링크는 저장소 README 참고). 이 노트북은 웹 접근이 어려운 교육
> 환경을 위한 **2순위 경로**로, 같은 프롬프트 쌍을 오픈 모델
> (Qwen2.5-1.5B-Instruct, Apache-2.0)로 실행한다.

같은 과제, 다른 지시 — 프롬프트가 "업무 지시서"임을
(X)/(O) 비교로 확인한다(12.3 설계 원칙 포맷 그대로).

### [준비] 환경 설정 (저장소 전용)

In [1]:
# Colab: 런타임 유형을 T4 GPU로 설정 후 실행
# !pip -q install transformers==4.44.2 accelerate
import os
os.environ["USE_TF"] = "0"      # PyTorch 경로 고정

import platform
import warnings
warnings.filterwarnings("ignore")   # 생성 설정 경고 정리(무해)
import transformers
transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()

print("Python", platform.python_version())
print("transformers", transformers.__version__)

Python 3.12.6
transformers 4.44.2


### [셀 1] 오픈 모델 로드와 질문 함수 *(경로 ②)* 📖

In [2]:
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer)
import torch

NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(
    NAME, torch_dtype=torch.float16,
    device_map="auto")

def ask(prompt, max_new=300):
    """프롬프트를 넣으면 응답 문자열을 돌려준다."""
    msgs = [{"role": "user", "content": prompt}]
    # apply_chat_template(tokenize=False)로 문자열을
    # 얻은 뒤 토크나이즈 — 버전 무관하게 동작하고
    # attention_mask까지 함께 넘긴다
    text = tok.apply_chat_template(
        msgs, tokenize=False,
        add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    # 그리디(do_sample=False) — 두 프롬프트 비교의 재현성
    out = model.generate(**ids, max_new_tokens=max_new,
                         do_sample=False)
    return tok.decode(
        out[0][ids["input_ids"].shape[1]:],
        skip_special_tokens=True).strip()

**핵심 포인트**
- 15억 파라미터의 소형 모델이라 Colab 무료 GPU(T4)에서 충분히 돈다. "무료로 끝까지"라는 이 실습의 설계다.
- `do_sample=False`(뽑기 끄기)로 두 프롬프트의 비교가 재현 가능하게 했다.

### [셀 2] 프롬프트 쌍 — 회의록 요약 과제 📖

In [3]:
MEETING = """3월 판촉 회의록입니다. 3월 5일 오후 2시, 마케팅팀 다섯 명이 참석했습니다.
이번 회의의 주제는 4월 판촉 방안입니다.
먼저 지난 분기 온라인 매출을 함께 살펴보았습니다.
매출이 목표에 조금 못 미쳤다는 점이 공유되었습니다.
특히 SNS 광고의 반응이 기대보다 낮았다는 의견이 있었습니다.
반면 신규 회원 가입은 꾸준히 늘고 있다는 이야기도 나왔습니다.
이런 배경에서 4월 광고 방식을 바꾸기로 뜻을 모았습니다.
논의 끝에, 4월 첫째 주부터 인스타그램 릴스 광고를 주 3회 올리기로 결정했습니다.
후속 작업으로, 박주임이 광고 영상 시안을 제작하기로 했습니다.
시안 완성 기한은 3월 20일입니다.
세부 예산과 문구는 추후 정하기로 했습니다."""

bad = f"이거 요약해 줘.\n{MEETING}"

good = f"""너는 회의록 담당자다(역할).
아래 회의록을 요약하라(과제).
- 신입사원이 읽어도 이해되게(조건: 독자)
- 결정 사항과 담당자를 빠뜨리지 말 것(조건)
- '결정/할 일/기한' 3항목 불릿으로(형식)
{MEETING}"""

print("=== Before ===\n", ask(bad))
print("=== After ===\n", ask(good))

=== Before ===
 ### 회의록 요약

#### 회의 일정 및 참석자
- **날짜**: 3월 5일
- **시간**: 오후 2시
- **참석자**: 마케팅팀 5명

#### 회의 주제
- **분야**: 4월 판촉 방안

#### 회의 내용
1. **분기 매출 분석**
   - **목표 비교**: 이전 분기에 비해 온라인 매출 목표에 미치지 못했다.
   - **SNS 광고 반응**: 기대보다 적은 반응을 보였다.

2. **방향 설정**
   - **광고 방식 변경**: SNS 광고의 성과가 저조하다는 점을 고려하여, 4월 첫째 주부터 인스타그램 릴스 광고를 주 3회씩 올릴 예정이다.

3. **추진 계획**
   - **영상을 제작하기 위한 협력**: 박주임이 광고 영상 시안을 제작할 예정이며, 시안 완성 기한은 3월 20일이다.
   - **예산 및 문구 정책**: 세부적인 예산과 문구는 추후 결정될 예정이다.

이 회의는 4


=== After ===
 **3월 판촉 회의록**

1. **회의 주제**: 4월 판촉 방안

2. **참석자**: 마케팅팀 다섯 명

3. **회의 내용**:
   - **분기별 매출 분석**: 지난 분기 온라인 매출을 함께 살펴보았으며, 목표에 조금 못 미치는 점이 발견되었다.
   - **SNS 광고 반응**: 특히 SNS 광고의 반응이 기대보다 낮았다는 의견이 나왔다.
   - **신규 회원 가입 증가**: 이와 같은 배경에서 4월 광고 방식을 변경하기로 결론 내렸다.
   - **광고 방식 결정**: 4월 첫째 주부터 인스타그램 릴스 광고를 주 3회 올리기로 결정했다.
   - **작업 계획**: 
     - 박주임이 광고 영상 시안을 제작하기로 했다.
     - 시안 완성 기한은 3월 20일이다.
     - 세부 예산과 문구는 추후 정하기로 했다.

4. **결정 사항 및 담당자**:
   - **광고 방식**: 인스타그램 �


**핵심 포인트**
- `good`의 구조가 〈프롬프트 기법〉에서 본 만능 템플릿의 네 칸이다. 괄호 주석까지 〈프롬프트 설계 원칙〉의 포맷 그대로다.
- 출력 대비의 실제: Before는 지시 없이 제멋대로 구성한 요약을, After는 "결정/할 일/기한"이 식별되는 요약을 낸다. 형식 지정이 출력을 눈에 띄게 바꾸는 것은 분명하나, 3항목 불릿을 문자 그대로 지키지 못하는 것은 소형 모델 지시 이행력의 한계다(스케일링의 체감판 — 경로 ①의 대형 모델은 훨씬 충실히 따른다).
- [보조 1] 셀에서 같은 과제에 퓨샷 예시와 CoT 지시를 얹어 한 단계 더 개선해 본다.

### [보조 1] 기법 확장 — 퓨샷과 사고 연쇄 (12.3 프롬프트 기법)

In [4]:
# 퓨샷: 원하는 출력의 예시를 1~2개 먼저 보여 준다
P_FEWSHOT = (
    "고객 문의를 분류해 줘.\n"
    "예시) '환불해 주세요' → 환불\n"
    "예시) '배송이 안 와요' → 배송\n"
    "문의: '카드 결제가 두 번 됐어요' →")
print(ask(P_FEWSHOT, max_new=16), "\n")

# 사고 연쇄: 풀이 과정을 밟게 한다
P_COT = ("사과 3개들이 봉지 4개를 사고 2개를 먹었다. "
         "남은 사과는? 단계별로 계산 과정을 보이고 "
         "마지막 줄에 '답: 숫자'로 답해 줘.")
print(ask(P_COT))

결제 



1. 처음에는 사과가 3개 있고, 봉지가 4개였습니다.
   - 사과의 수: 3개
   - 봉지의 수: 4개

2. 사과 중에서 2개를 먹었습니다.
   - 사과의 수: 3 - 2 = 1개

3. 남은 사과는 1개입니다.

결론:
남은 사과는 1개입니다.


### [심화 1] 만능 템플릿 빈칸 실험 (연습문제 응용 직결)

In [5]:
# 12.3 만능 템플릿 — 빈칸을 채워 나만의 지시서를
# 만들고, 요소를 하나씩 빼며 품질 변화를 기록하자.
TEMPLATE = """너는 {역할}이다.
{과제}
조건: {조건}
형식: {형식}"""

my_prompt = TEMPLATE.format(
    역할="여행 잡지 에디터",
    과제="제주도 2박 3일 일정을 짜 줘.",
    조건="대중교통만 이용, 하루 예산 10만 원",
    형식="날짜별 표(아침/점심/저녁 3열)")
print(ask(my_prompt, max_new=384))

죄송합니다, 저는 AI 어시스턴트로서 실제 여행 계획을 작성하는 능력이 없습니다. 하지만 제주도의 주요 관광지와 대중교통 정보를 제공해 드릴 수 있습니다. 아래는 제주도에서 2박 3일간의 일정 예시입니다:

**1일차**
- **오전**: 제주시청 - 제주국립공원 - 해양수산부관서 (1시간)
- **점심**: 제주국립공원 내 식당 - 간단한 점심 식사
- **오후**: 제주국립공원 내 자연경험 프로그램 (예: 바다생태체험 등)
- **저녁**: 제주국립공원 내 한식 요리집

**2일차**
- **오전**: 제주시청 - 제주도립미술관 - 제주도립박물관 (1시간)
- **점심**: 제주도립미술관 내 식당 - 간단한 점심 식사
- **오후**: 제주도립박물관 내 전시회 참가
- **저녁**: 제주도립박물관 내 한식 요리집

**3일차**
- **오전**: 제주시청 - 제주도립문화재단 - 제주도립민속박물관 (1시간)
- **점심**: 제주도립민속박물관 내 식당 - 간단한 점심 식사
- **오후**: 제주도립민속박물관 내 전시회 참가
- **저녁**: 제주도립민속박물관 내 한식 요리집

이


---
## 마무리

- 프롬프트는 **업무 지시서**다 — 역할·독자·조건·형식이 명확할수록 출력이 계약서처럼 돌아온다.
- (X)/(O) 비교는 어느 경로(웹 챗봇/오픈 모델)로 해도 성립한다 — 원칙은 모델을 가리지 않는다.
- 소형 모델의 한계 관찰까지가 실습이다 — 크기와 지시 이행력의 관계(12.2 스케일링).

**연습문제 연계**: [응용 필수] "(X) 프롬프트를 원칙 명시하며 (O)로 개선" 문항은 [심화 1] 템플릿에서 바로 연습한다.

**다음 실습**: [실습 12-3] RAG 미니 체험 — 저장소 전용 (`lab-12-03_rag-mini.ipynb`)